# Session 6 – Hierarchical Clustering on Iris & Dendrogram Interpretation

## 1. Hierarchical Clustering (Ward Linkage) on Iris + Dendrogram
The 4 original measurements (all in cm, so on a similar scale) are used directly.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

iris = load_iris()
X, y = iris.data, iris.target
print("Data shape:", X.shape)

Z_ward = linkage(X, method="ward")

plt.figure(figsize=(16, 7))
dendrogram(Z_ward, leaf_rotation=90, leaf_font_size=6, color_threshold=10)
plt.title("Iris Dataset – Hierarchical Clustering Dendrogram (Ward linkage)")
plt.xlabel("Sample index")
plt.ylabel("Distance (Ward)")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Truncated view - easier to read: show only the last 20 merges
plt.figure(figsize=(12, 6))
dendrogram(Z_ward, truncate_mode="lastp", p=20, show_contracted=True,
           leaf_rotation=45, leaf_font_size=10, color_threshold=10)
plt.title("Ward Dendrogram (last 20 merges) – numbers in brackets = samples in that branch")
plt.xlabel("Cluster (size)")
plt.ylabel("Distance (Ward)")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

print("Heights of the last 5 merges:", np.round(Z_ward[-5:, 2], 2))

## 2. Cutting the Dendrogram to Get 3 Clusters
From the dendrogram, the last merges happen at heights of about **6.4**, **12.3** and **32.4**.
Any horizontal cut **between 6.4 and 12.3** crosses exactly **3 vertical branches**, so we cut at **height = 10**.
The large vertical gap between 12.3 and 32.4 (and between 6.4 and 12.3) shows that these clusters are well separated.

In [ ]:
CUT_HEIGHT = 10

plt.figure(figsize=(16, 7))
dendrogram(Z_ward, leaf_rotation=90, leaf_font_size=6, color_threshold=CUT_HEIGHT)
plt.axhline(CUT_HEIGHT, color="red", linestyle="--", linewidth=2, label=f"Cut at height = {CUT_HEIGHT}")
plt.title("Ward Dendrogram cut into 3 clusters")
plt.xlabel("Sample index")
plt.ylabel("Distance (Ward)")
plt.legend()
plt.tight_layout()
plt.show()

clusters = fcluster(Z_ward, t=CUT_HEIGHT, criterion="distance")

print("Number of clusters:", len(np.unique(clusters)))
print("\nSamples in each cluster:")
print(pd.Series(clusters).value_counts().sort_index().rename_axis("Cluster").to_frame("Samples"))

In [ ]:
# How do the clusters compare with the real species?
print(pd.crosstab(clusters, iris.target_names[y], rownames=["Cluster"], colnames=["Species"]))

# Visualise on petal length vs petal width
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
axes[0].scatter(X[:, 2], X[:, 3], c=clusters, cmap="viridis", s=50, edgecolor="white")
axes[0].set_title("Hierarchical clusters (Ward, cut at 10)")
axes[1].scatter(X[:, 2], X[:, 3], c=y, cmap="viridis", s=50, edgecolor="white")
axes[1].set_title("Actual species")
for ax in axes:
    ax.set_xlabel("Petal length (cm)")
    ax.set_ylabel("Petal width (cm)")
    ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

**Result:** Cluster sizes are **50, 36 and 64**. The 50-sample cluster is exactly *setosa*; the other two mostly correspond to
*virginica* (36 cluster) and *versicolor* (64 cluster, which also absorbs ~15 virginica flowers that overlap with versicolor).

## 3. Ward vs Single vs Complete Linkage – Side-by-Side Dendrograms

In [ ]:
methods = ["ward", "single", "complete"]
linkages = {m: linkage(X, method=m) for m in methods}

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
for ax, m in zip(axes, methods):
    Zm = linkages[m]
    # colour threshold placed between the 2nd and 3rd last merges -> 3 coloured clusters
    thr = (Zm[-3, 2] + Zm[-2, 2]) / 2
    dendrogram(Zm, ax=ax, no_labels=True, color_threshold=thr)
    ax.axhline(thr, color="red", linestyle="--", linewidth=1.5)
    ax.set_title(f"{m.capitalize()} linkage (max height = {Zm[-1, 2]:.2f})", fontsize=13)
    ax.set_xlabel("Samples")
    ax.set_ylabel("Merge distance")
    ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.suptitle("Iris – Effect of Linkage Method on Dendrogram Structure", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Numerical comparison
rows = []
for m in methods:
    Zm = linkages[m]
    lbl = fcluster(Zm, t=3, criterion="maxclust")
    rows.append({
        "linkage": m,
        "last 3 merge heights": np.round(Zm[-3:, 2], 2).tolist(),
        "max height": round(Zm[-1, 2], 2),
        "cluster sizes (k=3)": sorted(np.bincount(lbl)[1:].tolist(), reverse=True),
    })
print(pd.DataFrame(rows).to_string(index=False))

for m in methods:
    lbl = fcluster(linkages[m], t=3, criterion="maxclust")
    print(f"\n{m.upper()} linkage - clusters vs species")
    print(pd.crosstab(lbl, iris.target_names[y], rownames=["Cluster"], colnames=["Species"]))

### Comparison
| Aspect | Ward | Single | Complete |
|---|---|---|---|
| **Merge heights** | Largest (top merge ≈ 32.4) – Ward measures increase in within-cluster variance, which grows quickly | Smallest (top merge ≈ 1.6) – only the closest pair of points matters | In between (top merge ≈ 7.1) – farthest pair of points |
| **Tree shape** | Compact, balanced branches with long vertical gaps at the top → clear cluster structure | Long "chaining" / staircase shape – points join one by one; very short gaps | Fairly balanced, tighter groups than single, but less clear gaps than Ward |
| **3-cluster sizes** | 50 / 64 / 36 – closely matches the 3 species | 98 / 50 / 2 – setosa separated, but versicolor + virginica are chained into one giant cluster, plus a tiny 2-point cluster | 72 / 50 / 28 – setosa separated, but versicolor & virginica split unevenly |
| **Verdict** | Best for Iris (roughly spherical, similar-sized groups) | Good at finding the clearly separated setosa, poor for overlapping groups | Better than single, but still mixes the overlapping species |

## 4. Interpreting the Dendrogram

- **Height = dissimilarity:** The height at which two branches join shows how different they are. Branches that merge **low** in the tree
  (short vertical lines) are **very similar** – e.g. the *versicolor* and *virginica* clusters merge at ≈ 12.3, so they are the most similar pair of clusters.
- **Highest merge = most different:** The *setosa* cluster joins the rest only at the very top (≈ 32.4), the tallest link in the tree – so setosa is the **most different** group.
- **Long vertical gaps = natural clusters:** A long vertical stretch with no merges (e.g. from ≈ 12.3 to 32.4) means the groups below it are well separated;
  cutting through the longest gaps gives the most natural number of clusters (here 2 or 3).
- **Structure:** Leaves that sit under the same small branch are near-duplicates, while the order of merging from bottom to top shows how individual flowers
  gradually form sub-groups, then species-level clusters.